# 🏈 NFL Quarterback Performance Analysis & Prediction
## A Deep-Dive into Passing Yards, Wins, and What Really Matters

**Authors:** Milan Jovkić, Uroš Petrašković  
**Course:** Analiza i Obrada Podataka  

---

### Project Overview

This notebook performs an **exhaustive exploratory data analysis** of NFL Quarterback performance spanning **1979–2025**, followed by a **machine learning pipeline** to predict future passing yards.

**Key Questions We Answer:**
1. How have QB passing stats evolved over the decades?
2. Which stats truly predict passing yards?
3. Do big passing numbers translate to wins — or is it just stat padding?
4. What features best predict *next-season* QB performance?
5. Who are the best QBs of 2025 — and could we have predicted it?

**Datasets Used:**
| Dataset | Records | Features | Source |
|---------|---------|----------|--------|
| QB Master Stats | 845 seasons | 142 columns | Pro Football Reference |
| QB ELO Rankings (Career) | 979 seasons | 46 columns | Custom ELO System |
| QB ELO Rankings (2025) | 63 QBs | 46 columns | Custom ELO System |
| Career vs 2025 Comparison | 63 QBs | 14 columns | Derived |

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Global plot styling
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 120,
    'font.size': 12,
    'axes.titlesize': 15,
    'axes.labelsize': 13,
    'legend.fontsize': 10,
    'figure.facecolor': 'white',
})

# Color palette for the project
COLORS = {
    'primary': '#013369',    # NFL Blue
    'secondary': '#D50A0A',  # NFL Red
    'accent': '#FFB612',     # Gold
    'green': '#2E8B57',
    'purple': '#6A0DAD',
    'orange': '#FF6B35',
    'teal': '#008080',
    'gray': '#708090',
}
NFL_PALETTE = ['#013369', '#D50A0A', '#FFB612', '#2E8B57', '#6A0DAD', '#FF6B35', '#008080', '#708090']

print("✅ All libraries loaded successfully!")

In [ ]:
# Load all datasets
qb = pd.read_csv('data/fully combined/qb_master.csv')
elo_career = pd.read_csv('data/nfl elo data/qb_rankings_career.csv')
elo_2025 = pd.read_csv('data/nfl elo data/qb_rankings_2025.csv')
comparison = pd.read_csv('data/nfl elo data/qb_career_vs_2025_comparison.csv')

# Parse win-loss records
def parse_record(rec):
    if pd.isna(rec):
        return np.nan, np.nan, np.nan
    parts = str(rec).split('-')
    if len(parts) == 3:
        return int(parts[0]), int(parts[1]), int(parts[2])
    return np.nan, np.nan, np.nan

qb[['Wins', 'Losses', 'Ties']] = qb['QBrec'].apply(lambda x: pd.Series(parse_record(x)))
qb['Win_Pct'] = qb['Wins'] / (qb['Wins'] + qb['Losses'] + qb['Ties'])
qb['TD_INT_Ratio'] = qb['TD'] / (qb['Int'].replace(0, 0.5))  # avoid div by zero

# Filter to meaningful seasons (starters with 8+ games started)
starters = qb[qb['GS'] >= 8].copy()

# Define eras for analysis
def assign_era(season):
    if season < 1990:
        return '1979-1989'
    elif season < 2000:
        return '1990-1999'
    elif season < 2010:
        return '2000-2009'
    elif season < 2020:
        return '2010-2019'
    else:
        return '2020-2025'

starters['Era'] = starters['Season'].apply(assign_era)
qb['Era'] = qb['Season'].apply(assign_era)

print(f"📊 QB Master Data: {qb.shape[0]} player-seasons, {qb.shape[1]} columns")
print(f"📊 Starters (8+ GS): {starters.shape[0]} player-seasons")
print(f"📊 Unique QBs: {starters['Player'].nunique()}")
print(f"📊 Seasons: {starters['Season'].min()} – {starters['Season'].max()}")
print(f"📊 ELO Career Data: {elo_career.shape[0]} entries")
print(f"📊 ELO 2025 Data: {elo_2025.shape[0]} entries")
print(f"\n📋 Key Columns: Yds, TD, Int, Rate, QBR, ANY/A, Y/A, Cmp%, Sk, AV, Win_Pct")

In [ ]:
# Quick data preview
display(starters[['Player', 'Season', 'Team', 'Age', 'G', 'GS', 'Cmp%', 'Yds', 'TD', 'Int', 
                   'Rate', 'QBR', 'ANY/A', 'Wins', 'Losses', 'Win_Pct']].sort_values('Yds', ascending=False).head(10))

## 2. League-Wide QB Trends Over Time
How has the quarterback position evolved? The NFL has been called a "passing league" — let's see if the data agrees.

In [ ]:
# Graph 1: Average Passing Yards Per Season
yearly = starters.groupby('Season').agg(
    avg_yds=('Yds', 'mean'),
    med_yds=('Yds', 'median'),
    total_yds=('Yds', 'sum'),
    n_qbs=('Player', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(yearly['Season'], yearly['avg_yds'], color=COLORS['primary'], linewidth=2.5, label='Mean Passing Yards', marker='o', markersize=4)
ax.plot(yearly['Season'], yearly['med_yds'], color=COLORS['secondary'], linewidth=2, linestyle='--', label='Median Passing Yards', alpha=0.8)
ax.fill_between(yearly['Season'], yearly['avg_yds'], alpha=0.15, color=COLORS['primary'])

# Add trend line
z = np.polyfit(yearly['Season'], yearly['avg_yds'], 2)
p = np.poly1d(z)
ax.plot(yearly['Season'], p(yearly['Season']), color=COLORS['accent'], linewidth=2, linestyle=':', label='Quadratic Trend')

ax.set_title('Average Passing Yards Per Season (Starters with 8+ GS)', fontsize=16, fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Passing Yards')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 2: Completion Percentage Over Time
yearly_cmp = starters.groupby('Season')['Cmp%'].agg(['mean', 'median', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(yearly_cmp['Season'], yearly_cmp['mean'], color=COLORS['green'], linewidth=2.5, marker='o', markersize=4, label='Mean Cmp%')
ax.fill_between(yearly_cmp['Season'], 
                yearly_cmp['mean'] - yearly_cmp['std'], 
                yearly_cmp['mean'] + yearly_cmp['std'], 
                alpha=0.2, color=COLORS['green'], label='±1 Std Dev')

ax.axhline(y=65, color=COLORS['secondary'], linestyle='--', alpha=0.5, label='65% Threshold (Modern Elite)')
ax.set_title('Evolution of Quarterback Accuracy: Completion % Over Time', fontsize=16, fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Completion Percentage (%)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 3: TD Rate vs INT Rate Over Time
yearly_rates = starters.groupby('Season').agg(
    avg_td_pct=('TD%', 'mean'),
    avg_int_pct=('Int%', 'mean')
).reset_index()

fig, ax1 = plt.subplots(figsize=(16, 7))

ax1.plot(yearly_rates['Season'], yearly_rates['avg_td_pct'], color=COLORS['green'], 
         linewidth=2.5, label='TD%', marker='o', markersize=4)
ax1.fill_between(yearly_rates['Season'], yearly_rates['avg_td_pct'], alpha=0.15, color=COLORS['green'])
ax1.set_ylabel('TD Rate (%)', color=COLORS['green'], fontsize=13)
ax1.tick_params(axis='y', labelcolor=COLORS['green'])

ax2 = ax1.twinx()
ax2.plot(yearly_rates['Season'], yearly_rates['avg_int_pct'], color=COLORS['secondary'], 
         linewidth=2.5, label='INT%', marker='s', markersize=4)
ax2.fill_between(yearly_rates['Season'], yearly_rates['avg_int_pct'], alpha=0.15, color=COLORS['secondary'])
ax2.set_ylabel('INT Rate (%)', color=COLORS['secondary'], fontsize=13)
ax2.tick_params(axis='y', labelcolor=COLORS['secondary'])

ax1.set_title('TD Rate vs Interception Rate: QBs Getting Smarter?', fontsize=16, fontweight='bold')
ax1.set_xlabel('Season')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=11)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 4: Average Passer Rating Over Time
yearly_rate = starters.groupby('Season')['Rate'].agg(['mean', 'median']).reset_index()

fig, ax = plt.subplots(figsize=(16, 7))
ax.bar(yearly_rate['Season'], yearly_rate['mean'], color=COLORS['primary'], alpha=0.6, width=0.8, label='Mean Rating')
ax.plot(yearly_rate['Season'], yearly_rate['median'], color=COLORS['secondary'], linewidth=2.5, label='Median Rating', marker='D', markersize=4)

# Rolling average
yearly_rate['rolling_5'] = yearly_rate['mean'].rolling(5, center=True).mean()
ax.plot(yearly_rate['Season'], yearly_rate['rolling_5'], color=COLORS['accent'], linewidth=3, label='5-Year Rolling Avg')

ax.set_title('Average Passer Rating by Season', fontsize=16, fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Passer Rating')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 5: Average Sacks Taken Per Season
yearly_sk = starters.groupby('Season')['Sk'].agg(['mean', 'sum']).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

ax1.plot(yearly_sk['Season'], yearly_sk['mean'], color=COLORS['purple'], linewidth=2.5, marker='o', markersize=4)
ax1.fill_between(yearly_sk['Season'], yearly_sk['mean'], alpha=0.2, color=COLORS['purple'])
ax1.set_title('Average Sacks Per QB Per Season', fontsize=14, fontweight='bold')
ax1.set_xlabel('Season')
ax1.set_ylabel('Sacks')
ax1.grid(True, alpha=0.3)

# Y/A over time
yearly_ya = starters.groupby('Season')['Y/A'].mean().reset_index()
ax2.plot(yearly_ya['Season'], yearly_ya['Y/A'], color=COLORS['orange'], linewidth=2.5, marker='o', markersize=4)
ax2.fill_between(yearly_ya['Season'], yearly_ya['Y/A'], alpha=0.2, color=COLORS['orange'])
ax2.set_title('Average Yards Per Attempt Over Time', fontsize=14, fontweight='bold')
ax2.set_xlabel('Season')
ax2.set_ylabel('Y/A')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Graph 6: Number of Qualifying QBs Over Time + Avg Attempts
yearly_counts = starters.groupby('Season').agg(
    n_qbs=('Player', 'count'),
    avg_att=('Att', 'mean')
).reset_index()

fig, ax1 = plt.subplots(figsize=(16, 7))

bars = ax1.bar(yearly_counts['Season'], yearly_counts['n_qbs'], color=COLORS['primary'], alpha=0.6, label='# QBs (8+ starts)')
ax1.set_ylabel('Number of Qualifying QBs', color=COLORS['primary'])
ax1.tick_params(axis='y', labelcolor=COLORS['primary'])

ax2 = ax1.twinx()
ax2.plot(yearly_counts['Season'], yearly_counts['avg_att'], color=COLORS['secondary'], linewidth=2.5, label='Avg Pass Attempts')
ax2.set_ylabel('Average Pass Attempts', color=COLORS['secondary'])
ax2.tick_params(axis='y', labelcolor=COLORS['secondary'])

ax1.set_title('Number of Starting QBs & Average Pass Attempts Per Season', fontsize=16, fontweight='bold')
ax1.set_xlabel('Season')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Statistical Distributions of QB Performance
Understanding the shape of QB performance data — who's average, who's elite, and how the game has shifted.

In [ ]:
# Graph 7: Distribution of Passing Yards
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Overall histogram
axes[0].hist(starters['Yds'], bins=40, color=COLORS['primary'], alpha=0.7, edgecolor='white')
axes[0].axvline(starters['Yds'].mean(), color=COLORS['secondary'], linestyle='--', linewidth=2, label=f"Mean: {starters['Yds'].mean():.0f}")
axes[0].axvline(starters['Yds'].median(), color=COLORS['accent'], linestyle='-.', linewidth=2, label=f"Median: {starters['Yds'].median():.0f}")
axes[0].set_title('Distribution of Season Passing Yards (All Eras)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Passing Yards')
axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=11)

# KDE by era
for era, color in zip(['1979-1989', '1990-1999', '2000-2009', '2010-2019', '2020-2025'],
                       [COLORS['gray'], COLORS['purple'], COLORS['teal'], COLORS['primary'], COLORS['secondary']]):
    subset = starters[starters['Era'] == era]['Yds']
    if len(subset) > 5:
        subset.plot.kde(ax=axes[1], label=era, linewidth=2.5, color=color)

axes[1].set_title('Passing Yards Distribution by Era (KDE)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Passing Yards')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=11)
axes[1].set_xlim(500, 6000)

plt.tight_layout()
plt.show()

In [ ]:
# Graph 8: Distribution of Passer Rating by Era
fig, ax = plt.subplots(figsize=(16, 7))

era_order = ['1979-1989', '1990-1999', '2000-2009', '2010-2019', '2020-2025']
era_colors = [COLORS['gray'], COLORS['purple'], COLORS['teal'], COLORS['primary'], COLORS['secondary']]

sns.violinplot(data=starters, x='Era', y='Rate', order=era_order, palette=era_colors, inner='box', alpha=0.8, ax=ax)
ax.set_title('Passer Rating Distribution by Era', fontsize=16, fontweight='bold')
ax.set_xlabel('Era')
ax.set_ylabel('Passer Rating')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 9: Box Plots of Key QB Stats by Era
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
era_order = ['1979-1989', '1990-1999', '2000-2009', '2010-2019', '2020-2025']
stats_to_plot = [
    ('Yds', 'Passing Yards'),
    ('TD', 'Touchdowns'),
    ('Cmp%', 'Completion %'),
    ('Y/A', 'Yards Per Attempt'),
    ('Int', 'Interceptions'),
    ('Sk', 'Sacks Taken')
]

for ax, (col, title) in zip(axes.flat, stats_to_plot):
    sns.boxplot(data=starters, x='Era', y=col, order=era_order, palette='RdYlBu_r', ax=ax)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Distribution of Key QB Stats Across Eras', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 10: Stacked KDE — How Passing Yards Shifted Decade by Decade
fig, ax = plt.subplots(figsize=(16, 8))

era_order = ['2020-2025', '2010-2019', '2000-2009', '1990-1999', '1979-1989']
era_colors = [COLORS['secondary'], COLORS['primary'], COLORS['teal'], COLORS['purple'], COLORS['gray']]

for i, (era, color) in enumerate(zip(era_order, era_colors)):
    subset = starters[starters['Era'] == era]['Yds'].dropna()
    if len(subset) > 5:
        from scipy.stats import gaussian_kde
        xs = np.linspace(500, 5500, 300)
        kde = gaussian_kde(subset)
        ys = kde(xs)
        ax.fill_between(xs, i * 0.0003 + ys, i * 0.0003, alpha=0.5, color=color, label=f'{era} (n={len(subset)}, μ={subset.mean():.0f})')
        ax.plot(xs, i * 0.0003 + ys, color=color, linewidth=1.5)

ax.set_title('Passing Yards Distribution Shifted Over Decades', fontsize=16, fontweight='bold')
ax.set_xlabel('Passing Yards')
ax.set_ylabel('Density (offset)')
ax.legend(fontsize=11, loc='upper right')
ax.set_yticks([])
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

## 4. Elite Quarterback Analysis
Who are the greatest passers in our dataset? How do career trajectories differ? What's the ideal QB age?

In [ ]:
# Graph 11: Top 15 Career Passing Yards Leaders
career_yds = starters.groupby('Player')['Yds'].sum().sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(career_yds.index, career_yds.values, color=COLORS['primary'], edgecolor='white', height=0.7)
for bar, val in zip(bars, career_yds.values):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2, f'{val:,.0f}', va='center', fontsize=11, fontweight='bold')
    
ax.set_title('Top 15 Career Passing Yards Leaders (8+ GS Seasons)', fontsize=16, fontweight='bold')
ax.set_xlabel('Career Passing Yards')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 12: Top 15 Career TD Leaders
career_td = starters.groupby('Player')['TD'].sum().sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(career_td.index, career_td.values, color=COLORS['secondary'], edgecolor='white', height=0.7)
for bar, val in zip(bars, career_td.values):
    ax.text(val + 3, bar.get_y() + bar.get_height()/2, f'{val}', va='center', fontsize=11, fontweight='bold')

ax.set_title('Top 15 Career Touchdown Leaders', fontsize=16, fontweight='bold')
ax.set_xlabel('Career Passing TDs')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 13: Career Passing Yards Trajectories — Top 8 QBs
top_8 = starters.groupby('Player')['Yds'].sum().sort_values(ascending=False).head(8).index.tolist()

fig, ax = plt.subplots(figsize=(16, 8))
for i, player in enumerate(top_8):
    pdf = starters[starters['Player'] == player].sort_values('Season')
    ax.plot(pdf['Season'], pdf['Yds'], marker='o', linewidth=2.5, markersize=5,
            label=player, color=NFL_PALETTE[i % len(NFL_PALETTE)])

ax.set_title('Career Passing Yards Trajectories — Top 8 All-Time Leaders', fontsize=16, fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Passing Yards')
ax.legend(fontsize=10, ncol=2, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 14: Career Passer Rating Trajectories — Top 8 QBs
fig, ax = plt.subplots(figsize=(16, 8))
for i, player in enumerate(top_8):
    pdf = starters[starters['Player'] == player].sort_values('Season')
    ax.plot(pdf['Season'], pdf['Rate'], marker='o', linewidth=2.5, markersize=5,
            label=player, color=NFL_PALETTE[i % len(NFL_PALETTE)])

ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='100 Rating Threshold')
ax.set_title('Passer Rating Trajectories — Top 8 All-Time Yardage Leaders', fontsize=16, fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Passer Rating')
ax.legend(fontsize=10, ncol=2, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 15: The QB Age Curve — Performance by Age
age_stats = starters.groupby('Age').agg(
    avg_yds=('Yds', 'mean'),
    avg_rate=('Rate', 'mean'),
    avg_td=('TD', 'mean'),
    avg_anya=('ANY/A', 'mean'),
    n=('Player', 'count')
).reset_index()
age_stats = age_stats[age_stats['n'] >= 5]  # need enough data

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Yards by age
axes[0,0].bar(age_stats['Age'], age_stats['avg_yds'], color=COLORS['primary'], alpha=0.7, edgecolor='white')
axes[0,0].set_title('Avg Passing Yards by Age', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Age')
axes[0,0].set_ylabel('Yards')
axes[0,0].axvline(x=age_stats.loc[age_stats['avg_yds'].idxmax(), 'Age'], color=COLORS['secondary'], linestyle='--', linewidth=2, label=f"Peak: Age {age_stats.loc[age_stats['avg_yds'].idxmax(), 'Age']:.0f}")
axes[0,0].legend()

# Rating by age
axes[0,1].bar(age_stats['Age'], age_stats['avg_rate'], color=COLORS['green'], alpha=0.7, edgecolor='white')
axes[0,1].set_title('Avg Passer Rating by Age', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('Age')
axes[0,1].set_ylabel('Passer Rating')
axes[0,1].axvline(x=age_stats.loc[age_stats['avg_rate'].idxmax(), 'Age'], color=COLORS['secondary'], linestyle='--', linewidth=2, label=f"Peak: Age {age_stats.loc[age_stats['avg_rate'].idxmax(), 'Age']:.0f}")
axes[0,1].legend()

# TDs by age
axes[1,0].bar(age_stats['Age'], age_stats['avg_td'], color=COLORS['orange'], alpha=0.7, edgecolor='white')
axes[1,0].set_title('Avg Touchdowns by Age', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Age')
axes[1,0].set_ylabel('Touchdowns')

# ANY/A by age  
axes[1,1].bar(age_stats['Age'], age_stats['avg_anya'], color=COLORS['purple'], alpha=0.7, edgecolor='white')
axes[1,1].set_title('Avg Adjusted Net Yards/Attempt by Age', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('Age')
axes[1,1].set_ylabel('ANY/A')

plt.suptitle('The QB Age Curve: When Do Quarterbacks Peak?', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 16: At What Age Do QBs Have Their Best Season?
# Best season = highest passer rating
peak_ages = starters.loc[starters.groupby('Player')['Rate'].idxmax()]['Age']

fig, ax = plt.subplots(figsize=(14, 7))
ax.hist(peak_ages.dropna(), bins=range(20, 42), color=COLORS['primary'], edgecolor='white', alpha=0.8)
ax.axvline(peak_ages.mean(), color=COLORS['secondary'], linestyle='--', linewidth=2.5, label=f'Mean Peak Age: {peak_ages.mean():.1f}')
ax.axvline(peak_ages.median(), color=COLORS['accent'], linestyle='-.', linewidth=2.5, label=f'Median Peak Age: {peak_ages.median():.1f}')
ax.set_title('Distribution of Peak Season Age (Best Passer Rating Season)', fontsize=16, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Number of QBs')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Passing Yards Deep Dive — What Drives Yardage?
What stats are most associated with high passing yards? Let's look at the ingredients.

In [ ]:
# Graph 17: Pass Attempts vs Passing Yards
fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(starters['Att'], starters['Yds'], c=starters['Rate'], cmap='RdYlGn', 
                     alpha=0.6, s=40, edgecolors='gray', linewidth=0.3)
cbar = plt.colorbar(scatter, ax=ax, label='Passer Rating')

# Add regression line
slope, intercept, r, p, se = stats.linregress(starters['Att'].dropna(), starters['Yds'].dropna())
x_line = np.linspace(starters['Att'].min(), starters['Att'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['secondary'], linewidth=2.5, 
        label=f'OLS Fit (R²={r**2:.3f})')

ax.set_title('Pass Attempts vs Passing Yards (colored by Passer Rating)', fontsize=16, fontweight='bold')
ax.set_xlabel('Pass Attempts')
ax.set_ylabel('Passing Yards')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 18: Completion % vs Passing Yards
fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(starters['Cmp%'], starters['Yds'], c=starters['Season'], cmap='viridis', 
                     alpha=0.6, s=40, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Season')

slope, intercept, r, p, se = stats.linregress(starters['Cmp%'].dropna(), starters.loc[starters['Cmp%'].notna(), 'Yds'])
x_line = np.linspace(starters['Cmp%'].min(), starters['Cmp%'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['secondary'], linewidth=2.5, 
        label=f'OLS Fit (R²={r**2:.3f})')

ax.set_title('Completion % vs Passing Yards (colored by Season)', fontsize=16, fontweight='bold')
ax.set_xlabel('Completion %')
ax.set_ylabel('Passing Yards')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 19: Yards Per Attempt vs Total Passing Yards
fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(starters['Y/A'], starters['Yds'], c=starters['Win_Pct'], cmap='RdYlGn',
                     alpha=0.6, s=40, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Win %')

ax.set_title('Y/A vs Passing Yards (colored by Win %)', fontsize=16, fontweight='bold')
ax.set_xlabel('Yards Per Attempt')
ax.set_ylabel('Passing Yards')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 20: Four Key Scatter Plots — Passing Yards Drivers
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# TD% vs Yards
axes[0,0].scatter(starters['TD%'], starters['Yds'], alpha=0.5, s=30, color=COLORS['green'])
axes[0,0].set_title('TD% vs Passing Yards', fontweight='bold')
axes[0,0].set_xlabel('TD%')
axes[0,0].set_ylabel('Passing Yards')
r_val = starters[['TD%', 'Yds']].dropna().corr().iloc[0,1]
axes[0,0].annotate(f'r = {r_val:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=12, fontweight='bold')

# Int% vs Yards
axes[0,1].scatter(starters['Int%'], starters['Yds'], alpha=0.5, s=30, color=COLORS['secondary'])
axes[0,1].set_title('INT% vs Passing Yards', fontweight='bold')
axes[0,1].set_xlabel('INT%')
axes[0,1].set_ylabel('Passing Yards')
r_val = starters[['Int%', 'Yds']].dropna().corr().iloc[0,1]
axes[0,1].annotate(f'r = {r_val:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=12, fontweight='bold')

# ANY/A vs Yards
axes[1,0].scatter(starters['ANY/A'], starters['Yds'], alpha=0.5, s=30, color=COLORS['purple'])
axes[1,0].set_title('ANY/A vs Passing Yards', fontweight='bold')
axes[1,0].set_xlabel('ANY/A')
axes[1,0].set_ylabel('Passing Yards')
r_val = starters[['ANY/A', 'Yds']].dropna().corr().iloc[0,1]
axes[1,0].annotate(f'r = {r_val:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=12, fontweight='bold')

# Sacks vs Yards
axes[1,1].scatter(starters['Sk'], starters['Yds'], alpha=0.5, s=30, color=COLORS['orange'])
axes[1,1].set_title('Sacks Taken vs Passing Yards', fontweight='bold')
axes[1,1].set_xlabel('Sacks')
axes[1,1].set_ylabel('Passing Yards')
r_val = starters[['Sk', 'Yds']].dropna().corr().iloc[0,1]
axes[1,1].annotate(f'r = {r_val:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=12, fontweight='bold')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle('Passing Yards: Key Stat Relationships', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 21: Correlation Heatmap — Passing Stats
pass_cols = ['Yds', 'TD', 'Int', 'Cmp%', 'Att', 'Rate', 'Y/A', 'AY/A', 'ANY/A', 'TD%', 
             'Int%', 'Sk', 'Y/G', 'QBR', '1D', 'Succ%', 'Sk%']
corr_data = starters[pass_cols].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            annot_kws={'size': 9})
ax.set_title('Correlation Heatmap: QB Passing Statistics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 22: Top Features Most Correlated with Passing Yards
numeric_cols = starters.select_dtypes(include=[np.number]).columns.tolist()
# Remove identifiers and target
exclude = ['Season', 'PlayerID', 'Yds', 'Wins', 'Losses', 'Ties', 'Win_Pct']
feature_cols = [c for c in numeric_cols if c not in exclude]

corr_with_yds = starters[feature_cols + ['Yds']].corr()['Yds'].drop('Yds').abs().sort_values(ascending=False)
top_30 = corr_with_yds.head(30)

fig, ax = plt.subplots(figsize=(14, 10))
colors = [COLORS['primary'] if v > 0.5 else COLORS['teal'] if v > 0.3 else COLORS['gray'] for v in top_30.values]
bars = ax.barh(range(len(top_30)), top_30.values, color=colors, edgecolor='white', height=0.7)
ax.set_yticks(range(len(top_30)))
ax.set_yticklabels(top_30.index, fontsize=10)
ax.invert_yaxis()
ax.set_title('Top 30 Features Correlated with Passing Yards (|r|)', fontsize=16, fontweight='bold')
ax.set_xlabel('Absolute Correlation')
ax.axvline(x=0.5, color=COLORS['secondary'], linestyle='--', alpha=0.5, label='Strong (>0.5)')
ax.axvline(x=0.3, color=COLORS['accent'], linestyle='--', alpha=0.5, label='Moderate (>0.3)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 6. Wins vs Stats — Are Big Numbers Just Stat Padding?

This is the critical question: **Do passing yards translate to wins?** A QB who throws for 4,500 yards but goes 4-13 is just stat-padding in garbage time. Let's separate the real performers from the empty calorie passers.

In [ ]:
# Graph 23: Passing Yards vs Win Percentage
valid = starters.dropna(subset=['Win_Pct', 'Yds'])

fig, ax = plt.subplots(figsize=(16, 9))
scatter = ax.scatter(valid['Yds'], valid['Win_Pct'], c=valid['Rate'], cmap='RdYlGn',
                     alpha=0.6, s=50, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Passer Rating')

# Add regression
slope, intercept, r, p, se = stats.linregress(valid['Yds'], valid['Win_Pct'])
x_line = np.linspace(valid['Yds'].min(), valid['Yds'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['secondary'], linewidth=2.5,
        label=f'OLS Fit (R²={r**2:.3f})')

# Label extreme outliers
high_yds_low_win = valid[(valid['Yds'] > 4000) & (valid['Win_Pct'] < 0.35)]
for _, row in high_yds_low_win.iterrows():
    ax.annotate(f"{row['Player']} ({row['Season']})", (row['Yds'], row['Win_Pct']),
                fontsize=8, alpha=0.8, textcoords="offset points", xytext=(5, 5))

ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='.500 Record')
ax.set_title('Passing Yards vs Win % — Do Big Numbers Mean Wins?', fontsize=16, fontweight='bold')
ax.set_xlabel('Passing Yards')
ax.set_ylabel('Win Percentage')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Correlation between Passing Yards and Win%: {valid['Yds'].corr(valid['Win_Pct']):.4f}")
print("→ Passing yards alone are a WEAK predictor of wins!")

In [ ]:
# Graph 24: Passer Rating vs Win Percentage
fig, ax = plt.subplots(figsize=(16, 9))
valid_rate = starters.dropna(subset=['Win_Pct', 'Rate'])

scatter = ax.scatter(valid_rate['Rate'], valid_rate['Win_Pct'], c=valid_rate['Season'], cmap='viridis',
                     alpha=0.6, s=50, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Season')

slope, intercept, r, p, se = stats.linregress(valid_rate['Rate'], valid_rate['Win_Pct'])
x_line = np.linspace(valid_rate['Rate'].min(), valid_rate['Rate'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['secondary'], linewidth=2.5,
        label=f'OLS Fit (R²={r**2:.3f})')

ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Passer Rating vs Win % — The Efficiency-Winning Connection', fontsize=16, fontweight='bold')
ax.set_xlabel('Passer Rating')
ax.set_ylabel('Win Percentage')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Correlation between Passer Rating and Win%: {valid_rate['Rate'].corr(valid_rate['Win_Pct']):.4f}")
print("→ Passer Rating is a MUCH BETTER predictor of wins than raw yards!")

In [ ]:
# Graph 25: ANY/A vs Win Percentage — The Best Single Stat?
valid_anya = starters.dropna(subset=['Win_Pct', 'ANY/A'])

fig, ax = plt.subplots(figsize=(16, 9))
scatter = ax.scatter(valid_anya['ANY/A'], valid_anya['Win_Pct'], c=valid_anya['Yds'], cmap='YlOrRd',
                     alpha=0.6, s=50, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Passing Yards')

slope, intercept, r, p, se = stats.linregress(valid_anya['ANY/A'], valid_anya['Win_Pct'])
x_line = np.linspace(valid_anya['ANY/A'].min(), valid_anya['ANY/A'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['primary'], linewidth=2.5,
        label=f'OLS Fit (R²={r**2:.3f})')

ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_title('ANY/A vs Win % — Adjusted Net Yards Per Attempt (colored by Total Yards)', fontsize=15, fontweight='bold')
ax.set_xlabel('ANY/A')
ax.set_ylabel('Win Percentage')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Correlation between ANY/A and Win%: {valid_anya['ANY/A'].corr(valid_anya['Win_Pct']):.4f}")
print("→ ANY/A (efficiency) is one of the best single stat predictors of winning!")

In [ ]:
# Graph 26: TD-INT Ratio vs Win %
valid_tdi = starters.dropna(subset=['Win_Pct', 'TD_INT_Ratio'])
valid_tdi = valid_tdi[valid_tdi['TD_INT_Ratio'] < 15]  # remove extreme outliers

fig, ax = plt.subplots(figsize=(16, 9))
scatter = ax.scatter(valid_tdi['TD_INT_Ratio'], valid_tdi['Win_Pct'], c=valid_tdi['Era'].map({
    '1979-1989': 0, '1990-1999': 1, '2000-2009': 2, '2010-2019': 3, '2020-2025': 4
}), cmap='viridis', alpha=0.6, s=50, edgecolors='gray', linewidth=0.3)

slope, intercept, r, p, se = stats.linregress(valid_tdi['TD_INT_Ratio'], valid_tdi['Win_Pct'])
x_line = np.linspace(0, valid_tdi['TD_INT_Ratio'].quantile(0.99), 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['secondary'], linewidth=2.5,
        label=f'OLS Fit (R²={r**2:.3f})')

ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_title('TD-to-INT Ratio vs Win % — Ball Security Matters', fontsize=16, fontweight='bold')
ax.set_xlabel('TD / INT Ratio')
ax.set_ylabel('Win Percentage')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Correlation: {valid_tdi['TD_INT_Ratio'].corr(valid_tdi['Win_Pct']):.4f}")

In [ ]:
# Graph 27: Comparison — Winning QBs vs Losing QBs
winners = starters[starters['Win_Pct'] >= 0.5].copy()
losers = starters[starters['Win_Pct'] < 0.5].copy()

compare_stats = ['Yds', 'TD', 'Int', 'Cmp%', 'Rate', 'Y/A', 'ANY/A', 'TD%', 'Int%', 'Sk']
winner_means = winners[compare_stats].mean()
loser_means = losers[compare_stats].mean()

fig, ax = plt.subplots(figsize=(16, 8))
x = np.arange(len(compare_stats))
width = 0.35

bars1 = ax.bar(x - width/2, winner_means, width, label=f'Winning QBs (≥.500, n={len(winners)})', 
               color=COLORS['green'], edgecolor='white', alpha=0.8)
bars2 = ax.bar(x + width/2, loser_means, width, label=f'Losing QBs (<.500, n={len(losers)})',
               color=COLORS['secondary'], edgecolor='white', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(compare_stats, fontsize=12)
ax.set_title('Average Stats: Winning QBs vs Losing QBs', fontsize=16, fontweight='bold')
ax.set_ylabel('Average Value')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{bar.get_height():.1f}', 
            ha='center', va='bottom', fontsize=8, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{bar.get_height():.1f}', 
            ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Insight: Winning QBs have COMPARABLE yards but BETTER efficiency stats!")
print(f"  Yards diff: {winner_means['Yds'] - loser_means['Yds']:.0f}")
print(f"  Rating diff: {winner_means['Rate'] - loser_means['Rate']:.1f}")
print(f"  ANY/A diff: {winner_means['ANY/A'] - loser_means['ANY/A']:.2f}")
print(f"  INT diff: {winner_means['Int'] - loser_means['Int']:.1f} (fewer is better)")

In [ ]:
# Graph 28: Win % by Passer Rating Tiers
valid_tiers = starters.dropna(subset=['Rate', 'Win_Pct']).copy()
valid_tiers['Rating_Tier'] = pd.cut(valid_tiers['Rate'], 
                                      bins=[0, 70, 80, 90, 100, 110, 160],
                                      labels=['<70', '70-80', '80-90', '90-100', '100-110', '110+'])

tier_stats = valid_tiers.groupby('Rating_Tier', observed=True).agg(
    avg_win_pct=('Win_Pct', 'mean'),
    n=('Player', 'count'),
    avg_yds=('Yds', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.bar(tier_stats['Rating_Tier'].astype(str), tier_stats['avg_win_pct'], 
              color=[COLORS['secondary'], COLORS['orange'], COLORS['accent'], COLORS['teal'], COLORS['green'], COLORS['primary']],
              edgecolor='white', alpha=0.85)

for bar, (_, row) in zip(bars, tier_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{row["avg_win_pct"]:.1%}\n(n={row["n"]})', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='.500')
ax.set_title('Average Win % by Passer Rating Tier', fontsize=16, fontweight='bold')
ax.set_xlabel('Passer Rating Tier')
ax.set_ylabel('Average Win %')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n→ Clear stepwise relationship: Higher passer rating = More wins!")
print("→ QBs with 110+ rating win nearly 70% of games")

In [ ]:
# Graph 29: Bubble Chart — Yards vs Wins (bubble = TDs, color = Rating)
valid_bubble = starters.dropna(subset=['Yds', 'Wins', 'TD', 'Rate']).copy()

fig, ax = plt.subplots(figsize=(16, 10))
scatter = ax.scatter(valid_bubble['Yds'], valid_bubble['Wins'], 
                     s=valid_bubble['TD'] * 8,  # bubble size = TDs
                     c=valid_bubble['Rate'], cmap='RdYlGn',
                     alpha=0.6, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Passer Rating')

# Annotate a few notable seasons
top_seasons = valid_bubble.nlargest(5, 'Yds')
for _, row in top_seasons.iterrows():
    ax.annotate(f"{row['Player']} ({int(row['Season'])})", 
                (row['Yds'], row['Wins']),
                fontsize=8, fontweight='bold', alpha=0.9,
                textcoords="offset points", xytext=(8, 3))

ax.set_title('Passing Yards vs Wins (Bubble Size = TDs, Color = Rating)', fontsize=16, fontweight='bold')
ax.set_xlabel('Passing Yards')
ax.set_ylabel('Wins')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 30: The "Stat Padders" vs "Efficient Winners" Quadrant Chart
valid_q = starters.dropna(subset=['Yds', 'Win_Pct', 'Rate']).copy()
yds_median = valid_q['Yds'].median()
win_median = 0.5

fig, ax = plt.subplots(figsize=(16, 10))
scatter = ax.scatter(valid_q['Yds'], valid_q['Win_Pct'], c=valid_q['Rate'], cmap='RdYlGn',
                     alpha=0.5, s=40, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Passer Rating')

# Draw quadrant lines
ax.axvline(x=yds_median, color='gray', linestyle='--', alpha=0.7)
ax.axhline(y=win_median, color='gray', linestyle='--', alpha=0.7)

# Label quadrants
ax.text(yds_median + 500, 0.85, '✅ ELITE\nHigh Yards + Winning', fontsize=12, fontweight='bold', color=COLORS['green'], ha='center')
ax.text(yds_median - 600, 0.85, '🤔 EFFICIENT\nLow Yards but Winning', fontsize=12, fontweight='bold', color=COLORS['primary'], ha='center')
ax.text(yds_median + 500, 0.15, '⚠️ STAT PADDERS\nHigh Yards but Losing', fontsize=12, fontweight='bold', color=COLORS['secondary'], ha='center')
ax.text(yds_median - 600, 0.15, '❌ STRUGGLING\nLow Yards + Losing', fontsize=12, fontweight='bold', color=COLORS['gray'], ha='center')

# Count and label each quadrant
elite = valid_q[(valid_q['Yds'] > yds_median) & (valid_q['Win_Pct'] >= win_median)]
efficient = valid_q[(valid_q['Yds'] <= yds_median) & (valid_q['Win_Pct'] >= win_median)]
stat_pad = valid_q[(valid_q['Yds'] > yds_median) & (valid_q['Win_Pct'] < win_median)]
struggling = valid_q[(valid_q['Yds'] <= yds_median) & (valid_q['Win_Pct'] < win_median)]

ax.text(yds_median + 500, 0.78, f'n={len(elite)}', fontsize=10, ha='center', color=COLORS['green'])
ax.text(yds_median - 600, 0.78, f'n={len(efficient)}', fontsize=10, ha='center', color=COLORS['primary'])
ax.text(yds_median + 500, 0.08, f'n={len(stat_pad)}', fontsize=10, ha='center', color=COLORS['secondary'])
ax.text(yds_median - 600, 0.08, f'n={len(struggling)}', fontsize=10, ha='center', color=COLORS['gray'])

ax.set_title('"Stat Padders" Quadrant: Yards vs Wins', fontsize=16, fontweight='bold')
ax.set_xlabel('Passing Yards')
ax.set_ylabel('Win Percentage')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f"\nQuadrant Breakdown:")
print(f"  ✅ Elite (High Yds + Winning): {len(elite)} ({len(elite)/len(valid_q)*100:.1f}%)")
print(f"  🤔 Efficient (Low Yds + Winning): {len(efficient)} ({len(efficient)/len(valid_q)*100:.1f}%)")
print(f"  ⚠️ Stat Padders (High Yds + Losing): {len(stat_pad)} ({len(stat_pad)/len(valid_q)*100:.1f}%)")
print(f"  ❌ Struggling (Low Yds + Losing): {len(struggling)} ({len(struggling)/len(valid_q)*100:.1f}%)")

In [ ]:
# Graph 31: Which Stat Correlates Best With Winning?
win_corrs = starters.dropna(subset=['Win_Pct'])[['Win_Pct', 'Yds', 'TD', 'Int', 'Cmp%', 'Rate', 
    'Y/A', 'AY/A', 'ANY/A', 'TD%', 'Int%', 'Sk', 'Y/G', 'TD_INT_Ratio', 
    'Y/C', 'QBR', '1D', 'Succ%']].corr()['Win_Pct'].drop('Win_Pct').sort_values()

fig, ax = plt.subplots(figsize=(14, 10))
colors = ['#D50A0A' if v < 0 else '#2E8B57' for v in win_corrs.values]
bars = ax.barh(win_corrs.index, win_corrs.values, color=colors, edgecolor='white', height=0.7)

for bar, val in zip(bars, win_corrs.values):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2, 
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')

ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('Correlation of QB Stats with Win Percentage', fontsize=16, fontweight='bold')
ax.set_xlabel('Pearson Correlation with Win %')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n🏆 Key Findings:")
print(f"  Best positive predictor: ANY/A ({win_corrs.get('ANY/A', 0):.3f})")
print(f"  Passer Rating: {win_corrs.get('Rate', 0):.3f}")
print(f"  Raw Yards: {win_corrs.get('Yds', 0):.3f} — NOT a strong predictor!")
print(f"  INT%: {win_corrs.get('Int%', 0):.3f} — Turnovers hurt wins")

## 7. QB ELO Analysis — Advanced Rating System

The ELO system provides a more nuanced ranking than traditional stats. Let's explore how ELO captures quarterback quality and compare career vs 2025 performance.

In [ ]:
# Graph 32: Distribution of QB ELO Ratings (Career)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].hist(elo_career['QB Elo'], bins=40, color=COLORS['primary'], alpha=0.7, edgecolor='white')
axes[0].axvline(elo_career['QB Elo'].mean(), color=COLORS['secondary'], linestyle='--', linewidth=2,
               label=f"Mean: {elo_career['QB Elo'].mean():.1f}")
axes[0].set_title('Distribution of Career QB ELO Ratings', fontsize=14, fontweight='bold')
axes[0].set_xlabel('QB ELO')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Scatter: ELO vs Win %
elo_career['Win_Pct'] = elo_career['W'] / (elo_career['W'] + elo_career['L'] + elo_career['T'].fillna(0))
axes[1].scatter(elo_career['QB Elo'], elo_career['Win_Pct'], alpha=0.4, s=30, color=COLORS['primary'])
slope, intercept, r, p, se = stats.linregress(elo_career['QB Elo'].dropna(), elo_career.loc[elo_career['QB Elo'].notna(), 'Win_Pct'])
x_line = np.linspace(elo_career['QB Elo'].min(), elo_career['QB Elo'].max(), 100)
axes[1].plot(x_line, slope * x_line + intercept, color=COLORS['secondary'], linewidth=2.5, label=f'R²={r**2:.3f}')
axes[1].set_title('QB ELO vs Win % (Career Seasons)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('QB ELO')
axes[1].set_ylabel('Win %')
axes[1].legend()
axes[1].axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 33: ELO vs QBR — How Do Rating Systems Compare?
elo_valid = elo_career.dropna(subset=['QB Elo', 'QBR', 'Passer Rtg'])

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].scatter(elo_valid['Passer Rtg'], elo_valid['QB Elo'], alpha=0.5, s=30, color=COLORS['primary'])
r1 = elo_valid['Passer Rtg'].corr(elo_valid['QB Elo'])
axes[0].set_title(f'Passer Rating vs QB ELO (r={r1:.3f})', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Passer Rating')
axes[0].set_ylabel('QB ELO')

axes[1].scatter(elo_valid['QBR'], elo_valid['QB Elo'], alpha=0.5, s=30, color=COLORS['green'])
r2 = elo_valid['QBR'].corr(elo_valid['QB Elo'])
axes[1].set_title(f'QBR vs QB ELO (r={r2:.3f})', fontsize=14, fontweight='bold')
axes[1].set_xlabel('ESPN QBR')
axes[1].set_ylabel('QB ELO')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 34: 2025 Season — Top 20 QBs by ELO Rating
top_elo = elo_2025.sort_values('QB Elo', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(14, 10))
colors = [COLORS['primary'] if elo > 200 else COLORS['teal'] if elo > 150 else COLORS['gray'] 
          for elo in top_elo['QB Elo']]
bars = ax.barh(top_elo['QB'], top_elo['QB Elo'], color=colors, edgecolor='white', height=0.7)
for bar, val in zip(bars, top_elo['QB Elo']):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2, f'{val:.1f}', va='center', fontsize=10, fontweight='bold')

ax.set_title('2025 Season: Top 20 QBs by ELO Rating', fontsize=16, fontweight='bold')
ax.set_xlabel('QB ELO')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 35: 2025 Season — Top 20 QBs by Passing Yards
top_yds_25 = elo_2025.sort_values('Yards', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(14, 10))
colors = [COLORS['green'] if w > l else COLORS['secondary'] for w, l in zip(top_yds_25['W'], top_yds_25['L'])]
bars = ax.barh(top_yds_25['QB'], top_yds_25['Yards'], color=colors, edgecolor='white', height=0.7)
for bar, val, w, l in zip(bars, top_yds_25['Yards'], top_yds_25['W'], top_yds_25['L']):
    ax.text(val + 30, bar.get_y() + bar.get_height()/2, f'{val:.0f} ({w}-{l})', va='center', fontsize=9, fontweight='bold')

ax.set_title('2025 Season: Top 20 Passing Yards Leaders (Green=Winning Record)', fontsize=15, fontweight='bold')
ax.set_xlabel('Passing Yards')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 36: 2025 Season — QBR vs Passing Yards
valid_25 = elo_2025.dropna(subset=['QBR', 'Yards'])

fig, ax = plt.subplots(figsize=(16, 10))
valid_25['Win_Pct'] = valid_25['W'] / (valid_25['W'] + valid_25['L'])
scatter = ax.scatter(valid_25['Yards'], valid_25['QBR'], 
                     c=valid_25['Win_Pct'], cmap='RdYlGn',
                     s=100, edgecolors='gray', linewidth=0.5, alpha=0.8)
plt.colorbar(scatter, ax=ax, label='Win %')

# Label all QBs
for _, row in valid_25.iterrows():
    ax.annotate(row['QB'], (row['Yards'], row['QBR']),
                fontsize=8, alpha=0.85, textcoords="offset points", xytext=(5, 5))

ax.set_title('2025 Season: QBR vs Passing Yards', fontsize=16, fontweight='bold')
ax.set_xlabel('Passing Yards')
ax.set_ylabel('ESPN QBR')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 37: Career Average ELO vs 2025 ELO — Who's Improving?
comp = comparison.dropna(subset=['Max_Elo', 'Elo_2025']).copy()
comp = comp[comp['Seasons_in_Career_Ranking'] >= 3]  # at least 3 career seasons

fig, ax = plt.subplots(figsize=(16, 10))
scatter = ax.scatter(comp['Max_Elo'], comp['Elo_2025'], 
                     s=comp['Career_Wins'] * 2, alpha=0.6,
                     c=comp['QBR_2025'], cmap='RdYlGn',
                     edgecolors='gray', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='2025 QBR')

# Diagonal line (career peak = 2025)
ax.plot([0, 350], [0, 350], 'k--', alpha=0.3, label='Career Peak = 2025 ELO')

for _, row in comp.iterrows():
    ax.annotate(row['QB'], (row['Max_Elo'], row['Elo_2025']),
                fontsize=8, alpha=0.85, textcoords="offset points", xytext=(5, 3))

ax.set_title('Career Peak ELO vs 2025 ELO (Bubble = Career Wins)', fontsize=15, fontweight='bold')
ax.set_xlabel('Career Max ELO')
ax.set_ylabel('2025 ELO')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Advanced Passing Metrics Deep Dive
Modern QB analysis goes beyond basic stats. Let's explore air yards, pressure rates, play action, and rushing contributions.

In [ ]:
# Graph 38: Air Yards vs Total Yards (Modern Era)
modern = starters[starters['Season'] >= 2018].dropna(subset=['adv_pass_pass_air_yds', 'Yds'])

fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(modern['adv_pass_pass_air_yds'], modern['Yds'], 
                     c=modern['Rate'], cmap='RdYlGn', s=60, alpha=0.7, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Passer Rating')

slope, intercept, r, p, se = stats.linregress(modern['adv_pass_pass_air_yds'], modern['Yds'])
x_line = np.linspace(modern['adv_pass_pass_air_yds'].min(), modern['adv_pass_pass_air_yds'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['secondary'], linewidth=2.5, label=f'R²={r**2:.3f}')

for _, row in modern.nlargest(5, 'Yds').iterrows():
    ax.annotate(f"{row['Player']} ({int(row['Season'])})", (row['adv_pass_pass_air_yds'], row['Yds']),
                fontsize=8, fontweight='bold', textcoords="offset points", xytext=(5, 5))

ax.set_title('Air Yards vs Total Passing Yards (2018+)', fontsize=16, fontweight='bold')
ax.set_xlabel('Air Yards')
ax.set_ylabel('Total Passing Yards')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 39: YAC vs Air Yards — How QBs Generate Yards
modern2 = starters[starters['Season'] >= 2018].dropna(subset=['adv_pass_pass_air_yds', 'adv_pass_pass_yac']).copy()
modern2['air_pct'] = modern2['adv_pass_pass_air_yds'] / (modern2['adv_pass_pass_air_yds'] + modern2['adv_pass_pass_yac'])

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Scatter: Air yards vs YAC
axes[0].scatter(modern2['adv_pass_pass_air_yds'], modern2['adv_pass_pass_yac'], 
                c=modern2['Win_Pct'], cmap='RdYlGn', s=50, alpha=0.6, edgecolors='gray', linewidth=0.3)
axes[0].set_title('Air Yards vs YAC (color = Win %)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Air Yards')
axes[0].set_ylabel('Yards After Catch (YAC)')
axes[0].grid(True, alpha=0.3)

# Distribution of Air Yards %
axes[1].hist(modern2['air_pct'].dropna(), bins=30, color=COLORS['primary'], alpha=0.7, edgecolor='white')
axes[1].axvline(modern2['air_pct'].mean(), color=COLORS['secondary'], linestyle='--', linewidth=2, 
                label=f"Mean: {modern2['air_pct'].mean():.1%}")
axes[1].set_title('Distribution of Air Yards % of Total', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Air Yards %')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Graph 40: Pressure Rate vs Passer Rating — Handling Pressure
modern3 = starters[starters['Season'] >= 2018].dropna(subset=['adv_pass_pass_pressured_pct', 'Rate']).copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Pressure % vs Rating
axes[0].scatter(modern3['adv_pass_pass_pressured_pct'] * 100, modern3['Rate'], 
                c=modern3['Win_Pct'], cmap='RdYlGn', s=50, alpha=0.6, edgecolors='gray')
r_val = modern3['adv_pass_pass_pressured_pct'].corr(modern3['Rate'])
axes[0].set_title(f'Pressure Rate vs Passer Rating (r={r_val:.3f})', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Pressured %')
axes[0].set_ylabel('Passer Rating')
axes[0].grid(True, alpha=0.3)

# Poor Throw % vs Rating
modern4 = starters[starters['Season'] >= 2018].dropna(subset=['adv_pass_pass_poor_throw_pct', 'Rate'])
axes[1].scatter(modern4['adv_pass_pass_poor_throw_pct'] * 100, modern4['Rate'], 
                c=modern4['Win_Pct'], cmap='RdYlGn', s=50, alpha=0.6, edgecolors='gray')
r_val2 = modern4['adv_pass_pass_poor_throw_pct'].corr(modern4['Rate'])
axes[1].set_title(f'Poor Throw % vs Passer Rating (r={r_val2:.3f})', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Poor Throw %')
axes[1].set_ylabel('Passer Rating')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Graph 41: Play Action & RPO Analysis
modern5 = starters[starters['Season'] >= 2018].copy()
pa_cols = ['adv_pass_pass_play_action', 'adv_pass_pass_play_action_pass_yds',
           'adv_pass_pass_rpo', 'adv_pass_pass_rpo_yds']
modern5 = modern5.dropna(subset=pa_cols)

# Play Action usage vs Yds
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

axes[0,0].scatter(modern5['adv_pass_pass_play_action'], modern5['adv_pass_pass_play_action_pass_yds'],
                  c=modern5['Rate'], cmap='RdYlGn', s=50, alpha=0.6)
axes[0,0].set_title('Play Action Plays vs Play Action Yards', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Play Action Count')
axes[0,0].set_ylabel('Play Action Yards')

# PA yards as % of total
modern5['pa_pct'] = modern5['adv_pass_pass_play_action_pass_yds'] / modern5['Yds']
axes[0,1].hist(modern5['pa_pct'].dropna(), bins=25, color=COLORS['purple'], alpha=0.7, edgecolor='white')
axes[0,1].set_title('Distribution of Play Action Yards % of Total', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Play Action Yds %')

# RPO analysis
axes[1,0].scatter(modern5['adv_pass_pass_rpo'], modern5['adv_pass_pass_rpo_yds'],
                  c=modern5['Rate'], cmap='RdYlGn', s=50, alpha=0.6)
axes[1,0].set_title('RPO Plays vs RPO Yards', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('RPO Count')
axes[1,0].set_ylabel('RPO Yards')

# Scramble yards contribution
modern6 = starters[starters['Season'] >= 2018].dropna(subset=['adv_pass_rush_scrambles', 'rr_Rush_Yds'])
axes[1,1].scatter(modern6['adv_pass_rush_scrambles'], modern6['rr_Rush_Yds'],
                  c=modern6['Win_Pct'], cmap='RdYlGn', s=50, alpha=0.6)
axes[1,1].set_title('Scrambles vs Total Rush Yards (color = Win%)', fontsize=13, fontweight='bold')
axes[1,1].set_xlabel('Scrambles')
axes[1,1].set_ylabel('Rush Yards')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle('Advanced Passing: Play Action, RPO & Scrambling', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 42: Dual-Threat QBs — Does Rushing Help Win?
rush_data = starters.dropna(subset=['rr_Rush_Yds', 'Win_Pct']).copy()
rush_data['Rush_Tier'] = pd.cut(rush_data['rr_Rush_Yds'], bins=[0, 100, 300, 500, 2000],
                                  labels=['<100', '100-300', '300-500', '500+'])

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Rush yards vs passing yards
axes[0].scatter(rush_data['rr_Rush_Yds'], rush_data['Yds'], c=rush_data['Win_Pct'], 
                cmap='RdYlGn', s=40, alpha=0.5, edgecolors='gray')
axes[0].set_title('Rush Yards vs Passing Yards (color=Win%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rushing Yards')
axes[0].set_ylabel('Passing Yards')
axes[0].grid(True, alpha=0.3)

# Win % by rush tier
tier_wins = rush_data.groupby('Rush_Tier', observed=True)['Win_Pct'].agg(['mean', 'count']).reset_index()
bars = axes[1].bar(tier_wins['Rush_Tier'].astype(str), tier_wins['mean'], 
                   color=[COLORS['gray'], COLORS['teal'], COLORS['primary'], COLORS['green']], 
                   edgecolor='white', alpha=0.8)
for bar, (_, row) in zip(bars, tier_wins.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{row["mean"]:.1%}\n(n={int(row["count"])})', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Win % by QB Rushing Tier', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Rushing Yards Tier')
axes[1].set_ylabel('Average Win %')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Predicting Next-Season QB Performance
The core of our analysis: Can we predict a QB's future passing yards? We build features from current-season stats to predict next-season performance.

In [ ]:
# Feature Engineering: Create next-season target variable
# Sort by player and season
qb_sorted = starters.sort_values(['Player', 'Season']).copy()

# Create next-season yards as target
qb_sorted['Next_Yds'] = qb_sorted.groupby('Player')['Yds'].shift(-1)
qb_sorted['Next_TD'] = qb_sorted.groupby('Player')['TD'].shift(-1)
qb_sorted['Next_Rate'] = qb_sorted.groupby('Player')['Rate'].shift(-1)
qb_sorted['Next_Wins'] = qb_sorted.groupby('Player')['Wins'].shift(-1)

# Drop rows without a next season (last season for each player)
model_data = qb_sorted.dropna(subset=['Next_Yds']).copy()

# Select features (current season stats to predict next season)
feature_cols = ['Age', 'G', 'GS', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'Int', 'TD%', 'Int%',
                '1D', 'Succ%', 'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate', 'Sk', 'Sk%', 
                'NY/A', 'ANY/A', 'AV', 'Win_Pct', 'Wins', 'TD_INT_Ratio']

# Also add rushing if available
rush_features = ['rr_Rush_Att', 'rr_Rush_Yds', 'rr_Rush_TD', 'rr_Rush_Y/A']
for col in rush_features:
    if col in model_data.columns:
        feature_cols.append(col)

# Clean data
X = model_data[feature_cols].copy()
y = model_data['Next_Yds'].copy()

# Impute missing values
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_clean = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

print(f"📊 Model Dataset: {len(X_clean)} samples, {len(feature_cols)} features")
print(f"📊 Target: Next-season passing yards")
print(f"📊 Feature columns: {feature_cols}")
print(f"\n📋 Target distribution:")
print(f"  Mean: {y.mean():.0f}")
print(f"  Std: {y.std():.0f}")
print(f"  Min: {y.min():.0f}, Max: {y.max():.0f}")

In [ ]:
# Graph 43: Current Season Yards vs Next Season Yards
fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(model_data['Yds'], model_data['Next_Yds'], c=model_data['Age'], 
                     cmap='viridis', s=40, alpha=0.5, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Age')

# Perfect prediction line
max_val = max(model_data['Yds'].max(), model_data['Next_Yds'].max())
ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Perfect Prediction')

# Regression line
slope, intercept, r, p, se = stats.linregress(model_data['Yds'], model_data['Next_Yds'])
x_line = np.linspace(model_data['Yds'].min(), model_data['Yds'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['secondary'], linewidth=2.5, label=f'OLS Fit (R²={r**2:.3f})')

ax.set_title('Current Season Yards vs Next Season Yards', fontsize=16, fontweight='bold')
ax.set_xlabel('Current Season Passing Yards')
ax.set_ylabel('Next Season Passing Yards')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nYear-over-year yards correlation: {r:.4f}")
print("→ Moderate correlation — other factors matter!")

In [ ]:
# Graph 44: Which Current-Season Stats Best Predict NEXT-Season Yards?
corr_next = pd.concat([X_clean, y], axis=1).corr()['Next_Yds'].drop('Next_Yds').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 10))
colors = [COLORS['primary'] if v > 0.5 else COLORS['green'] if v > 0.3 else COLORS['gray'] for v in corr_next.values]
bars = ax.barh(range(len(corr_next)), corr_next.values, color=colors, edgecolor='white', height=0.7)
ax.set_yticks(range(len(corr_next)))
ax.set_yticklabels(corr_next.index, fontsize=10)
ax.invert_yaxis()
ax.set_title('Predicting NEXT Season Yards: Feature Correlations', fontsize=16, fontweight='bold')
ax.set_xlabel('|Correlation| with Next Season Yards')
ax.axvline(x=0.5, color=COLORS['secondary'], linestyle='--', alpha=0.5, label='Strong')
ax.axvline(x=0.3, color=COLORS['accent'], linestyle='--', alpha=0.5, label='Moderate')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 45: Scatter Matrix — Top 6 Predictive Features
top_features = corr_next.head(6).index.tolist()
scatter_data = model_data[top_features + ['Next_Yds']].dropna()

fig, axes = plt.subplots(2, 3, figsize=(20, 13))
for ax, feat in zip(axes.flat, top_features):
    ax.scatter(scatter_data[feat], scatter_data['Next_Yds'], alpha=0.4, s=25, color=COLORS['primary'])
    r_val = scatter_data[feat].corr(scatter_data['Next_Yds'])
    ax.set_title(f'{feat} (r={r_val:.3f})', fontsize=12, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Next Season Yards')
    ax.grid(True, alpha=0.3)
    # Add trend line
    slope, intercept, _, _, _ = stats.linregress(scatter_data[feat], scatter_data['Next_Yds'])
    x_line = np.linspace(scatter_data[feat].min(), scatter_data[feat].max(), 100)
    ax.plot(x_line, slope * x_line + intercept, color=COLORS['secondary'], linewidth=2)

plt.suptitle('Top 6 Features for Predicting Next-Season Passing Yards', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 10. Model Training & Comparison
We train multiple regression models and compare their performance using time-series cross-validation.

In [ ]:
# Train/Test Split (chronological — last 2 seasons as test)
train_mask = model_data['Season'] <= 2023
test_mask = model_data['Season'] > 2023

X_train = X_clean[train_mask]
X_test = X_clean[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Train set: {len(X_train)} samples (up to 2023)")
print(f"Test set: {len(X_test)} samples (2024-2025)")
print(f"Features: {X_train.shape[1]}")

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=10.0),
    'Lasso': Lasso(alpha=1.0, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000),
    'KNN': KNeighborsRegressor(n_neighbors=7, weights='distance'),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Cross-validation on training set
    cv = TimeSeriesSplit(n_splits=5)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error')
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_MAE': -cv_scores.mean()}
    predictions[name] = y_pred
    print(f"  {name:25s} | MAE={mae:7.1f} | RMSE={rmse:7.1f} | R²={r2:.4f} | CV MAE={-cv_scores.mean():.1f}")

results_df = pd.DataFrame(results).T
print("\n✅ All models trained!")

In [ ]:
# Graph 46: Model Comparison — MAE, RMSE, R²
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# MAE
axes[0].barh(results_df.index, results_df['MAE'], color=NFL_PALETTE[:len(results_df)], edgecolor='white')
axes[0].set_title('Mean Absolute Error (lower is better)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('MAE (yards)')
for i, v in enumerate(results_df['MAE']):
    axes[0].text(v + 5, i, f'{v:.0f}', va='center', fontsize=10)

# RMSE
axes[1].barh(results_df.index, results_df['RMSE'], color=NFL_PALETTE[:len(results_df)], edgecolor='white')
axes[1].set_title('Root Mean Squared Error (lower is better)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('RMSE (yards)')
for i, v in enumerate(results_df['RMSE']):
    axes[1].text(v + 5, i, f'{v:.0f}', va='center', fontsize=10)

# R²
axes[2].barh(results_df.index, results_df['R2'], color=NFL_PALETTE[:len(results_df)], edgecolor='white')
axes[2].set_title('R² Score (higher is better)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('R²')
for i, v in enumerate(results_df['R2']):
    axes[2].text(max(v + 0.01, 0.01), i, f'{v:.3f}', va='center', fontsize=10)

plt.suptitle('Model Comparison: Predicting Next-Season Passing Yards', fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 47: Actual vs Predicted — Best Model
best_model_name = results_df['MAE'].idxmin()
best_preds = predictions[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Actual vs Predicted scatter
axes[0].scatter(y_test, best_preds, c=COLORS['primary'], s=60, alpha=0.7, edgecolors='gray')
max_val = max(y_test.max(), best_preds.max())
min_val = min(y_test.min(), best_preds.min())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_title(f'Actual vs Predicted ({best_model_name})', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Actual Next-Season Yards')
axes[0].set_ylabel('Predicted Next-Season Yards')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Residual plot
residuals = y_test.values - best_preds
axes[1].scatter(best_preds, residuals, c=COLORS['primary'], s=60, alpha=0.7, edgecolors='gray')
axes[1].axhline(y=0, color=COLORS['secondary'], linewidth=2)
axes[1].set_title(f'Residual Plot ({best_model_name})', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Yards')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best Model: {best_model_name}")
print(f"  MAE: {results[best_model_name]['MAE']:.1f} yards")
print(f"  RMSE: {results[best_model_name]['RMSE']:.1f} yards")
print(f"  R²: {results[best_model_name]['R2']:.4f}")

In [ ]:
# Graph 48: Prediction Error Distribution
fig, ax = plt.subplots(figsize=(14, 7))

for name, preds in predictions.items():
    errors = y_test.values - preds
    ax.hist(errors, bins=20, alpha=0.4, label=f'{name} (μ={errors.mean():.0f}, σ={errors.std():.0f})')

ax.axvline(x=0, color='black', linewidth=2, linestyle='--')
ax.set_title('Prediction Error Distribution (All Models)', fontsize=16, fontweight='bold')
ax.set_xlabel('Error (Actual - Predicted, yards)')
ax.set_ylabel('Frequency')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 49: Feature Importance — Random Forest
rf_model = models['Random Forest']
rf_model.fit(X_train_scaled, y_train)
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(14, 12))
colors = [COLORS['primary'] if v > importances.quantile(0.75) else COLORS['teal'] if v > importances.median() else COLORS['gray'] 
          for v in importances.values]
ax.barh(importances.index, importances.values, color=colors, edgecolor='white', height=0.7)
ax.set_title('Feature Importance — Random Forest', fontsize=16, fontweight='bold')
ax.set_xlabel('Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n🏆 Top 10 Most Important Features for Predicting Next-Season Yards:")
for i, (feat, imp) in enumerate(importances.tail(10).iloc[::-1].items()):
    print(f"  {i+1}. {feat}: {imp:.4f}")

In [ ]:
# Graph 50: Feature Importance — Gradient Boosting
gb_model = models['Gradient Boosting']
gb_model.fit(X_train_scaled, y_train)
gb_imp = pd.Series(gb_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(14, 12))
colors = [COLORS['green'] if v > gb_imp.quantile(0.75) else COLORS['teal'] if v > gb_imp.median() else COLORS['gray'] 
          for v in gb_imp.values]
ax.barh(gb_imp.index, gb_imp.values, color=colors, edgecolor='white', height=0.7)
ax.set_title('Feature Importance — Gradient Boosting', fontsize=16, fontweight='bold')
ax.set_xlabel('Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# Compare top features across both models
print("\n📊 Top 5 Features Comparison:")
print(f"  Random Forest: {list(importances.tail(5).index[::-1])}")
print(f"  Gradient Boosting: {list(gb_imp.tail(5).index[::-1])}")

## 11. SHAP Analysis — Explaining the Model
SHAP (SHapley Additive exPlanations) tells us exactly *how* each feature contributes to individual predictions.

In [ ]:
# Graph 51-53: SHAP Analysis
try:
    import shap
    
    # Use the best tree-based model
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test_scaled)
    
    # SHAP Summary Plot
    fig, ax = plt.subplots(figsize=(14, 10))
    shap.summary_plot(shap_values, X_test_scaled, plot_type='bar', show=False, max_display=20)
    plt.title('SHAP Feature Importance (Mean |SHAP|)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"SHAP analysis skipped: {e}")
    print("Continuing with other analyses...")

In [ ]:
# Graph 52: SHAP Beeswarm Plot
try:
    fig, ax = plt.subplots(figsize=(14, 10))
    shap.summary_plot(shap_values, X_test_scaled, show=False, max_display=15)
    plt.title('SHAP Beeswarm: How Features Push Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"SHAP beeswarm skipped: {e}")

In [ ]:
# Graph 53: SHAP Dependence Plots for Top Features
try:
    top_shap_features = pd.Series(np.abs(shap_values).mean(axis=0), index=X_test_scaled.columns).nlargest(4).index.tolist()
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    for ax, feat in zip(axes.flat, top_shap_features):
        shap.dependence_plot(feat, shap_values, X_test_scaled, ax=ax, show=False)
        ax.set_title(f'SHAP Dependence: {feat}', fontsize=12, fontweight='bold')
    
    plt.suptitle('SHAP Dependence Plots — Top 4 Features', fontsize=17, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"SHAP dependence plots skipped: {e}")

## 12. 2025 Season Deep Dive
Let's zoom into the current 2025 season — who's performing, who's overperforming, and what the data tells us.

In [ ]:
# Graph 54: 2025 Season — Top 15 QBs by ELO Points (Performance Score)
top15 = elo_2025.nlargest(15, 'Points')

fig, ax = plt.subplots(figsize=(14, 8))
colors = [COLORS['green'] if p > 1.5 else COLORS['primary'] if p > 0.5 else COLORS['orange'] if p > 0 else COLORS['secondary']
          for p in top15['Points']]
bars = ax.barh(range(len(top15)), top15['Points'], color=colors, edgecolor='white', height=0.7)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['QB'])
ax.invert_yaxis()
for bar, val in zip(bars, top15['Points']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=10)
ax.set_title('2025 Season: Top 15 QBs by ELO Performance Points', fontsize=16, fontweight='bold')
ax.set_xlabel('Points')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 55: 2025 Season — Efficiency vs Volume Scatter
fig, ax = plt.subplots(figsize=(16, 10))
e25 = elo_2025.copy()
e25['Win_Pct'] = e25['W'] / (e25['W'] + e25['L'])

scatter = ax.scatter(e25['Yards'], e25['ANY/A'], 
                     c=e25['Win_Pct'], cmap='RdYlGn',
                     s=e25['TDs'] * 10, alpha=0.7, edgecolors='gray', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Win %')

for _, row in e25.nlargest(20, 'Points').iterrows():
    ax.annotate(row['QB'], (row['Yards'], row['ANY/A']),
                fontsize=8, fontweight='bold', alpha=0.9,
                textcoords="offset points", xytext=(5, 5))

ax.set_title('2025 Season: Yards vs ANY/A (Bubble=TDs, Color=Win%)', fontsize=16, fontweight='bold')
ax.set_xlabel('Passing Yards')
ax.set_ylabel('ANY/A')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 56: 2025 Season — Completion % vs Yards Per Attempt
fig, ax = plt.subplots(figsize=(16, 10))
e25 = elo_2025.copy()
e25['Win_Pct'] = e25['W'] / (e25['W'] + e25['L'])

scatter = ax.scatter(e25['Comp%'] * 100, e25['YPA'],
                     c=e25['Points'], cmap='coolwarm', s=80, alpha=0.7, edgecolors='gray')
plt.colorbar(scatter, ax=ax, label='ELO Points')

for _, row in e25.nlargest(15, 'Points').iterrows():
    ax.annotate(row['QB'], (row['Comp%'] * 100, row['YPA']),
                fontsize=8, fontweight='bold', textcoords="offset points", xytext=(5, 5))

ax.set_title('2025 Season: Completion % vs Yards Per Attempt', fontsize=16, fontweight='bold')
ax.set_xlabel('Completion %')
ax.set_ylabel('Yards Per Attempt')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 57: 2025 Season — TD% vs INT% (Ball Security)
fig, ax = plt.subplots(figsize=(16, 10))
e25 = elo_2025.copy()
e25['Win_Pct'] = e25['W'] / (e25['W'] + e25['L'])

scatter = ax.scatter(e25['TD%'] * 100, e25['INT%'] * 100,
                     c=e25['Win_Pct'], cmap='RdYlGn', s=80, alpha=0.7, edgecolors='gray')
plt.colorbar(scatter, ax=ax, label='Win %')

for _, row in e25.nlargest(20, 'Points').iterrows():
    ax.annotate(row['QB'], (row['TD%'] * 100, row['INT%'] * 100),
                fontsize=8, fontweight='bold', textcoords="offset points", xytext=(5, 5))

# Ideal quadrant
ax.axvline(x=e25['TD%'].median() * 100, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=e25['INT%'].median() * 100, color='gray', linestyle='--', alpha=0.5)
ax.text(0.95, 0.05, 'High TD%, Low INT%\n= Best', transform=ax.transAxes, fontsize=11,
        fontweight='bold', color=COLORS['green'], ha='right', va='bottom')

ax.set_title('2025 Season: TD% vs INT% (color = Win%)', fontsize=16, fontweight='bold')
ax.set_xlabel('TD%')
ax.set_ylabel('INT%')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Year-Over-Year Consistency — Who's Reliable?
A great QB isn't just flashy one season — they're consistent year after year.

In [ ]:
# Graph 58: Year-Over-Year Passing Yards Consistency
multi_season = starters.groupby('Player').filter(lambda x: len(x) >= 5)
player_std = multi_season.groupby('Player').agg(
    mean_yds=('Yds', 'mean'),
    std_yds=('Yds', 'std'),
    seasons=('Season', 'count'),
    mean_rate=('Rate', 'mean'),
    mean_win=('Win_Pct', 'mean')
).reset_index()
player_std['cv'] = player_std['std_yds'] / player_std['mean_yds']  # Coefficient of variation

fig, ax = plt.subplots(figsize=(16, 10))
scatter = ax.scatter(player_std['mean_yds'], player_std['cv'], 
                     c=player_std['mean_win'], cmap='RdYlGn', s=player_std['seasons'] * 20,
                     alpha=0.7, edgecolors='gray', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Avg Win %')

for _, row in player_std.nlargest(10, 'mean_yds').iterrows():
    ax.annotate(row['Player'], (row['mean_yds'], row['cv']),
                fontsize=9, fontweight='bold', textcoords="offset points", xytext=(5, 5))
for _, row in player_std.nsmallest(3, 'cv').iterrows():
    if row['mean_yds'] > 2500:
        ax.annotate(row['Player'], (row['mean_yds'], row['cv']),
                    fontsize=9, fontweight='bold', color=COLORS['green'],
                    textcoords="offset points", xytext=(5, -10))

ax.set_title('QB Consistency: Avg Yards vs Coefficient of Variation (Bubble=Seasons)', fontsize=15, fontweight='bold')
ax.set_xlabel('Average Passing Yards Per Season')
ax.set_ylabel('Coefficient of Variation (lower = more consistent)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 59: Year-Over-Year Passer Rating Stability
fig, ax = plt.subplots(figsize=(16, 10))
player_std2 = multi_season.groupby('Player').agg(
    mean_rate=('Rate', 'mean'),
    std_rate=('Rate', 'std'),
    seasons=('Season', 'count'),
    mean_win=('Win_Pct', 'mean')
).reset_index()
player_std2['cv_rate'] = player_std2['std_rate'] / player_std2['mean_rate']

scatter = ax.scatter(player_std2['mean_rate'], player_std2['cv_rate'],
                     c=player_std2['mean_win'], cmap='RdYlGn', s=player_std2['seasons'] * 20,
                     alpha=0.7, edgecolors='gray', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Avg Win %')

for _, row in player_std2.nlargest(10, 'mean_rate').iterrows():
    ax.annotate(row['Player'], (row['mean_rate'], row['cv_rate']),
                fontsize=9, fontweight='bold', textcoords="offset points", xytext=(5, 5))

ax.set_title('QB Rating Consistency: Avg Rating vs CV (Bubble=Seasons)', fontsize=15, fontweight='bold')
ax.set_xlabel('Average Passer Rating')
ax.set_ylabel('Coefficient of Variation (lower = more consistent)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Advanced Visualizations — Heatmaps & Radar Charts

In [ ]:
# Graph 60: Performance Heatmap — Average Passer Rating by Age × Era
pivot = starters.pivot_table(values='Rate', index='Age', columns='Era', aggfunc='mean')
pivot = pivot.reindex(columns=['1979-1989', '1990-1999', '2000-2009', '2010-2019', '2020-2025'])
pivot = pivot.loc[22:40]  # restrict to common ages

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=80, linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Average Passer Rating by Age × Era', fontsize=16, fontweight='bold')
ax.set_xlabel('Era')
ax.set_ylabel('Age')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 61: Passing Yards Heatmap — by Age × Era
pivot_yds = starters.pivot_table(values='Yds', index='Age', columns='Era', aggfunc='mean')
pivot_yds = pivot_yds.reindex(columns=['1979-1989', '1990-1999', '2000-2009', '2010-2019', '2020-2025'])
pivot_yds = pivot_yds.loc[22:40]

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot_yds, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Average Passing Yards by Age × Era', fontsize=16, fontweight='bold')
ax.set_xlabel('Era')
ax.set_ylabel('Age')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 62: Radar Chart — Top 5 QBs of 2025
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches

e25 = elo_2025.copy()
top5 = e25.nlargest(5, 'Points')

# Normalize stats to 0-1 for radar
radar_cols = ['Comp%', 'YPA', 'TD%', 'Success', 'ANY/A']
radar_labels = ['Comp%', 'YPA', 'TD%', 'Success Rate', 'ANY/A']

# Normalize
for col in radar_cols:
    min_val = e25[col].min()
    max_val = e25[col].max()
    top5[col + '_norm'] = (top5[col] - min_val) / (max_val - min_val)

angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
angles += angles[:1]  # close the circle

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(polar=True))
colors_radar = [COLORS['primary'], COLORS['secondary'], COLORS['green'], COLORS['purple'], COLORS['orange']]

for i, (_, row) in enumerate(top5.iterrows()):
    values = [row[col + '_norm'] for col in radar_cols]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2.5, label=row['QB'], color=colors_radar[i], markersize=6)
    ax.fill(angles, values, alpha=0.1, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=12)
ax.set_title('Top 5 QBs of 2025: Performance Radar', fontsize=16, fontweight='bold', y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 63: Cross-Validation Scores Distribution
fig, ax = plt.subplots(figsize=(16, 8))

cv_results = {}
for name, model in models.items():
    cv = TimeSeriesSplit(n_splits=5)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error')
    cv_results[name] = -scores

bp = ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True)
for patch, color in zip(bp['boxes'], NFL_PALETTE[:len(models)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Cross-Validation MAE Distribution (5-Fold Time Series)', fontsize=16, fontweight='bold')
ax.set_ylabel('MAE (yards)')
ax.tick_params(axis='x', rotation=25)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Graph 64: Learning Curve — Does More Data Help?
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (name, model) in zip(axes, [('Random Forest', models['Random Forest']), 
                                       ('Gradient Boosting', models['Gradient Boosting'])]):
    train_sizes, train_scores, test_scores = learning_curve(
        model, X_train_scaled, y_train, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.2, 1.0, 8), scoring='neg_mean_absolute_error'
    )
    
    train_mean = -train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    test_mean = -test_scores.mean(axis=1)
    test_std = test_scores.std(axis=1)
    
    ax.plot(train_sizes, train_mean, 'o-', color=COLORS['primary'], label='Training MAE')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color=COLORS['primary'])
    ax.plot(train_sizes, test_mean, 'o-', color=COLORS['secondary'], label='Validation MAE')
    ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color=COLORS['secondary'])
    
    ax.set_title(f'Learning Curve: {name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('MAE (yards)')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 15. Individual QB Deep Dives — Career Spotlight Charts

In [ ]:
# Graph 65: Career Profile Cards — Top 6 QBs
top_6_qbs = starters.groupby('Player')['Yds'].sum().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(22, 14))

for ax, player in zip(axes.flat, top_6_qbs):
    pdf = starters[starters['Player'] == player].sort_values('Season')
    
    ax.bar(pdf['Season'], pdf['Yds'], color=COLORS['primary'], alpha=0.7, edgecolor='white')
    ax2 = ax.twinx()
    ax2.plot(pdf['Season'], pdf['Rate'], color=COLORS['secondary'], linewidth=2.5, marker='o', markersize=5)
    ax2.set_ylabel('Rating', color=COLORS['secondary'], fontsize=10)
    ax2.tick_params(axis='y', labelcolor=COLORS['secondary'])
    
    career_yds = pdf['Yds'].sum()
    career_wins = pdf['Wins'].sum()
    career_losses = pdf['Losses'].sum()
    ax.set_title(f'{player}\n{career_yds:,.0f} yds | {career_wins:.0f}-{career_losses:.0f}', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Season', fontsize=9)
    ax.set_ylabel('Yards', color=COLORS['primary'], fontsize=10)
    ax.tick_params(axis='y', labelcolor=COLORS['primary'])
    ax.grid(True, alpha=0.2)

plt.suptitle('Career Profiles: Top 6 QBs (Bars=Yards, Line=Passer Rating)', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Graph 66: Career TD% vs INT% — All-Time Efficiency Leaders
career_eff = starters.groupby('Player').agg(
    avg_td_pct=('TD%', 'mean'),
    avg_int_pct=('Int%', 'mean'),
    total_yds=('Yds', 'sum'),
    seasons=('Season', 'count'),
    avg_win=('Win_Pct', 'mean')
).reset_index()
career_eff = career_eff[career_eff['seasons'] >= 4]  # at least 4 qualifying seasons

fig, ax = plt.subplots(figsize=(16, 10))
scatter = ax.scatter(career_eff['avg_td_pct'], career_eff['avg_int_pct'], 
                     c=career_eff['avg_win'], cmap='RdYlGn', s=career_eff['total_yds'] / 200,
                     alpha=0.7, edgecolors='gray', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Avg Win %')

# Label notable QBs
for _, row in career_eff.nlargest(12, 'total_yds').iterrows():
    ax.annotate(row['Player'], (row['avg_td_pct'], row['avg_int_pct']),
                fontsize=9, fontweight='bold', textcoords="offset points", xytext=(5, 5))

ax.axhline(y=career_eff['avg_int_pct'].median(), color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=career_eff['avg_td_pct'].median(), color='gray', linestyle='--', alpha=0.5)

ax.set_title('Career Avg TD% vs INT% (Bubble=Total Yards, Color=Win%)', fontsize=16, fontweight='bold')
ax.set_xlabel('Average TD%')
ax.set_ylabel('Average INT%')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 16. Cumulative Career Analysis

In [ ]:
# Graph 67: Cumulative Career Passing Yards — Race Chart
top_8_cum = starters.groupby('Player')['Yds'].sum().nlargest(8).index.tolist()

fig, ax = plt.subplots(figsize=(16, 9))
for i, player in enumerate(top_8_cum):
    pdf = starters[starters['Player'] == player].sort_values('Season')
    pdf['cum_yds'] = pdf['Yds'].cumsum()
    ax.plot(pdf['Season'], pdf['cum_yds'], marker='o', linewidth=2.5, markersize=4,
            label=f"{player} ({pdf['cum_yds'].iloc[-1]:,.0f})", color=NFL_PALETTE[i % len(NFL_PALETTE)])

ax.set_title('Cumulative Career Passing Yards — Top 8 QBs', fontsize=16, fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Cumulative Passing Yards')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
# Graph 68: Cumulative Career Wins — Top 8 QBs
fig, ax = plt.subplots(figsize=(16, 9))
for i, player in enumerate(top_8_cum):
    pdf = starters[starters['Player'] == player].sort_values('Season')
    pdf['cum_wins'] = pdf['Wins'].cumsum()
    ax.plot(pdf['Season'], pdf['cum_wins'], marker='o', linewidth=2.5, markersize=4,
            label=f"{player} ({pdf['cum_wins'].iloc[-1]:.0f}W)", color=NFL_PALETTE[i % len(NFL_PALETTE)])

ax.set_title('Cumulative Career Wins — Top 8 QBs', fontsize=16, fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Cumulative Wins')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 17. Summary of Key Findings

In [ ]:
# Graph 69: Summary — What Stats Actually Matter for QB Success
# Final correlation comparison: Stats vs Yards and Stats vs Wins
key_stats = ['Cmp%', 'Y/A', 'AY/A', 'ANY/A', 'TD%', 'Int%', 'Rate', 'QBR', 'Sk%', 'Succ%', 'TD_INT_Ratio']
valid_summary = starters.dropna(subset=['Win_Pct'] + key_stats)

corr_yds = valid_summary[key_stats + ['Yds']].corr()['Yds'][key_stats]
corr_wins = valid_summary[key_stats + ['Win_Pct']].corr()['Win_Pct'][key_stats]

fig, ax = plt.subplots(figsize=(16, 8))
x = np.arange(len(key_stats))
width = 0.35

bars1 = ax.bar(x - width/2, corr_yds, width, label='Corr with Passing Yards', color=COLORS['primary'], alpha=0.8)
bars2 = ax.bar(x + width/2, corr_wins, width, label='Corr with Win %', color=COLORS['green'], alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(key_stats, rotation=30, ha='right', fontsize=11)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_title('What Matters: Correlation with Yards vs Correlation with Wins', fontsize=16, fontweight='bold')
ax.set_ylabel('Pearson Correlation')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("🏆 KEY INSIGHT: Stats that predict YARDS ≠ Stats that predict WINS")
print("=" * 70)
print("• Raw volume stats (Att, Cmp) drive YARDS but not WINS")
print("• Efficiency stats (ANY/A, Rate, TD%) drive WINS more than YARDS")
print("• INT% has NEGATIVE correlation with BOTH — turnovers hurt everything")
print("• The best QBs have high efficiency AND high volume")

In [ ]:
# Graph 70: The Ultimate QB Success Formula — What Predicts What
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

valid_final = starters.dropna(subset=['ANY/A', 'Win_Pct', 'Yds', 'Rate'])

# ANY/A vs Yards colored by wins
sc1 = axes[0].scatter(valid_final['ANY/A'], valid_final['Yds'], c=valid_final['Win_Pct'], 
                       cmap='RdYlGn', s=30, alpha=0.5)
axes[0].set_title('ANY/A vs Yards\n(color = Win%)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('ANY/A')
axes[0].set_ylabel('Passing Yards')

# Rate vs Wins colored by yards
sc2 = axes[1].scatter(valid_final['Rate'], valid_final['Win_Pct'], c=valid_final['Yds'],
                       cmap='YlOrRd', s=30, alpha=0.5)
axes[1].set_title('Rating vs Win%\n(color = Yards)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Passer Rating')
axes[1].set_ylabel('Win %')

# TD% - INT% vs Win%
valid_final['TD_minus_INT'] = valid_final['TD%'] - valid_final['Int%']
sc3 = axes[2].scatter(valid_final['TD_minus_INT'], valid_final['Win_Pct'], c=valid_final['Rate'],
                       cmap='coolwarm', s=30, alpha=0.5)
axes[2].set_title('TD%-INT% vs Win%\n(color = Rating)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('TD% minus INT%')
axes[2].set_ylabel('Win %')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle('The QB Success Formula: Efficiency > Volume', fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 18. Conclusions

### Key Findings

**1. The NFL IS a Passing League — And It's Getting More So**
- Average passing yards per starter have increased from ~2,800 in the 1980s to ~3,800+ today
- Completion percentages have risen from ~55% to ~65%+
- Interception rates have dropped significantly

**2. Passing Yards Alone Don't Win Games**
- Raw passing yards have a **weak correlation** with winning (~0.15-0.20)
- Many high-yardage QBs are simply throwing a lot in losing situations ("garbage time")
- The "Stat Padders" quadrant analysis shows this clearly

**3. Efficiency Stats Predict Wins Much Better**
- **ANY/A** (Adjusted Net Yards Per Attempt) is the single best correlator with wins
- **Passer Rating** is also strongly predictive
- **TD-to-INT ratio** matters significantly
- QBs in the 110+ passer rating tier win nearly 70% of their games

**4. Predicting Next-Season Performance**
- Current-season passing attempts and yards are the strongest predictors of next-season yards
- However, efficiency metrics improve prediction accuracy
- Tree-based models (Random Forest, Gradient Boosting) outperform linear models
- The best model achieves meaningful predictive power for next-season yards

**5. The QB Age Curve**
- QBs peak between ages 27-31 for both volume and efficiency
- There is significant variation in peak age across individuals
- Modern QBs tend to sustain performance longer

**6. Advanced Metrics Add Value**
- Air Yards, YAC distribution, pressure handling, and play action usage all provide meaningful signal
- Dual-threat ability (rushing) provides a slight edge in winning

### The Bottom Line
> **The best QBs combine volume AND efficiency. High passing yards with low efficiency = stat padding. The true elite are those with high ANY/A, high passer rating, AND winning records.**